# EDA 15 (test): kiểm tra trên nhóm test (D19), v1 và v2 cạnh nhau

**Chỉ chạy một lần, sau khi khoá cấu hình.** Nhóm test (D17) chưa được dùng để chọn bất kỳ ngưỡng nào. Notebook này chạy bước 7 cho nhóm test với **cả hai cách gom địa điểm** (v1 và v2, D20), đo **đúng những thước đo đã dùng trên validation** và đặt các nhóm cạnh nhau.

| Mục | Đo gì | So với validation |
|---|---|---|
| T1 | Cấu hình đang khoá (in từ code) | ghi ngày khoá vào D19 |
| T2 | Có home / office, các cờ, confidence, cho v1 và v2 | bảng 5a, 5b (notebook 13) |
| T3 | Nửa đầu / nửa sau có ra cùng nơi theo mức confidence, cho v1 và v2 | bảng 4 (notebook 13) |
| T4 | Stay nhận nhầm theo nhãn phương tiện (không phụ thuộc v1 / v2) | bảng 4 (notebook 10) |
| T5 | v1 và v2 có cùng home / office không; gộp nhầm mới ở v2 | bảng 2, 4b (notebook 14) |

**Quy tắc (D19):** không sửa code hay ngưỡng sau khi xem số ở đây. Lệch nhiều so với validation thì ghi là hạn chế. Nếu vẫn sửa thì số test này không còn là số test.

**Hạn chế:** đo độ ổn định và mức đồng thuận giữa v1 với v2 trên user mới, không đo độ đúng vì không có nhãn home / office (A13). Không dùng F1.

> **Giai đoạn: TEST (chạy một lần).** Chỉ chạy sau khi đã khoá cấu hình (D19). Không sửa gì sau khi xem số.
>
> Cần có sẵn `data/processed/places_v1` và `places_v2` (chạy `cluster_v1` và `cluster_v2` cho mọi user). Nếu `reports/home_office_<v>_test.csv` đã có thì chỉ đọc lại, không chạy đè.

In [1]:
import importlib
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from preprocess import interim
from stays import detect
from geo import haversine_m
from places import cluster_v1, cluster_v2, common
from places import home_office_conf as hoc
from eda import split, stay_scan as ss, stay_anchor as sa, conf_holdout as ch, place_eps as pe
for m in (hoc, ch):
    importlib.reload(m)

VERSIONS = ["v1", "v2"]
validation, test = split.val_users(), split.test_users()
print(f"validation {len(validation)} user, test {len(test)} user, trùng nhau: {len(set(validation) & set(test))}")

validation 115 user, test 57 user, trùng nhau: 0


## T1. Cấu hình đang khoá

In thẳng từ code để chắc chắn test chạy đúng cấu hình đã chốt. Nếu khác D19 thì dừng lại.

In [2]:
T1 = pd.Series({
    "neo stay": detect.METHOD,
    "D (m)": detect.DIST_M,
    "T (phút)": detect.TIME_S / 60,
    "max gap (giờ)": detect.MAX_GAP_S / 3600,
    "v1: bán kính gom địa điểm (m)": cluster_v1.RADIUS_M,
    "v2: eps (m)": cluster_v2.EPS_M,
    "v2: min_samples": cluster_v2.MIN_SAMPLES,
    "khung đêm (giờ)": hoc.NIGHT,
    "khung làm việc (giờ)": hoc.WORK,
    "phút tối thiểu mỗi phiếu": hoc.MIN_VOTE_MIN,
    "mốc phiếu (đêm / ngày làm việc)": (hoc.N_MIN_NIGHTS, hoc.N_MIN_WORKDAYS),
    "số lần bốc lại": hoc.N_BOOT,
    "hạt giống": hoc.SEED,
}, name="giá trị")
display(T1.to_frame())

,giá trị
neo stay,h1_mean
D (m),200.0
T (phút),30.0
max gap (giờ),24.0
v1: bán kính gom địa điểm (m),200.0
v2: eps (m),200.0
v2: min_samples,1
khung đêm (giờ),"(20.0, 31.0)"
khung làm việc (giờ),"[(9.0, 18.0)]"
phút tối thiểu mỗi phiếu,60.0


## T2. Kết quả bước 7 trên nhóm test

Chạy `home_office_conf` cho các user test với từng version và ghi `reports/home_office_<v>_test.csv` (giống lệnh `python -m src.places.home_office_conf --test --version <v>`). **Nếu file đã có thì chỉ đọc lại, không chạy đè**, để số test là của lần chạy đầu tiên. Muốn chạy lại phải xoá file bằng tay và ghi lý do vào D19.

In [3]:
ho = {}
for v in VERSIONS:
    test_csv = ROOT / "reports" / f"home_office_{v}_test.csv"
    dev_csv = ROOT / "reports" / f"home_office_{v}_val.csv"
    if test_csv.exists():
        print(f"Đã có {test_csv.name}: chỉ đọc lại, không chạy đè.")
    else:
        test_csv.parent.mkdir(parents=True, exist_ok=True)
        hoc.run(test, common.places_dir(v)).to_csv(test_csv, index=False)
        print(f"Đã chạy và ghi {test_csv.name}")
    ho[(v, "validation")] = pd.read_csv(dev_csv, dtype={"user": str})
    ho[(v, "test")] = pd.read_csv(test_csv, dtype={"user": str})
name = lambda k: f"{k[0]} · {k[1]}"

def tong_quan(d):
    n = len(d)
    f = d["flags"].fillna("")
    pct = lambda m: round(100 * m.sum() / n, 1)
    return {"số user": n, "có home (%)": pct(d["home"].notna()), "có office (%)": pct(d["office"].notna()),
            "có cả hai (%)": pct(d["home"].notna() & d["office"].notna()),
            "home = office (%)": pct(f.str.contains("home_eq_office")),
            "không cờ nào (%)": pct(f == "")}

print("Bảng T2a. Tổng quan")
display(pd.DataFrame({name(k): tong_quan(v) for k, v in ho.items()}))
print("Bảng T2b. % user theo cờ")
display(pd.DataFrame({name(k): (100 * v["flags"].fillna("").str.get_dummies(";").sum() / len(v)).round(1) for k, v in ho.items()}).fillna(0))
print("Bảng T2c. Confidence (%)")
display(pd.concat({name(k): hoc.conf_summary(v) for k, v in ho.items()}, axis=1))

Đã có home_office_v1_test.csv: chỉ đọc lại, không chạy đè.
Đã có home_office_v2_test.csv: chỉ đọc lại, không chạy đè.
Bảng T2a. Tổng quan


,v1 · validation,v1 · test,v2 · validation,v2 · test
số user,115.0,57.0,115.0,57.0
có home (%),87.0,94.7,87.0,94.7
có office (%),48.7,63.2,46.1,52.6
có cả hai (%),45.2,61.4,42.6,50.9
home = office (%),26.1,26.3,28.7,36.8
không cờ nào (%),20.9,29.8,21.7,21.1


Bảng T2b. % user theo cờ


,v1 · validation,v1 · test,v2 · validation,v2 · test
few_nights,42.6,49.1,42.6,49.1
few_workdays,51.3,49.1,51.3,49.1
home_eq_office,26.1,26.3,28.7,36.8
no_night,7.0,3.5,7.0,3.5
no_stays,6.1,1.8,6.1,1.8
no_workday,19.1,8.8,19.1,8.8
unstable_home,6.1,7.0,2.6,7.0
unstable_office,2.6,7.0,0.0,5.3


Bảng T2c. Confidence (%)


v1 · validation        v1 · test         \
                                      home office      home office   
có kết quả                             100     56        54     36   
có confidence (đủ 7 phiếu)              59     33        28     22   
trung vị (%)                          89.2   96.2      97.4   83.8   
dưới 70%                                19      8         5      7   
70–90%                                  11      6         3      5   
từ 90%                                  29     19        20     10   

                           v2 · validation        v2 · test         
                                      home office      home office  
có kết quả                             100     53        54     30  
có confidence (đủ 7 phiếu)              59     31        28     16  
trung vị (%)                          98.0  100.0      99.8   95.9  
dưới 70%                                15      3         3      2  
70–90%                                   8      7         2      4  
từ 90%                                  36     21        23     10

**Đọc kết quả:** nếu tỉ lệ có home / office, các cờ và trung vị confidence của test gần validation (chênh vài điểm % với mẫu nhỏ là bình thường) thì phương pháp không chỉ khớp riêng nhóm validation, và đúng với cả v1 lẫn v2. Chênh lớn thì xem có phải do test có nhiều user ít dữ liệu hơn không (cờ `few_*`). Confidence của v2 cao hơn v1 một phần là do gộp cụm (cơ học), không phải bằng chứng v2 đúng hơn (notebook 14).

## T3. Nửa đầu / nửa sau theo mức confidence

Giống mục 4 của notebook 13: chia stay của từng user làm hai nửa theo giờ đến, bỏ phiếu riêng, hỏi user có confidence nửa đầu cao thì nửa sau có ra cùng nơi nhiều hơn không. Dev chạy lại tại đây để hai bảng cùng một code.

In [4]:
R = {(v, g): ch.run(users, common.places_dir(v)) for v in VERSIONS for g, users in (("validation", validation), ("test", test))}
for kind in ("home", "office"):
    out = {}
    for k, r in R.items():
        g, rho, n = ch.summary(r, kind)
        out[name(k)] = g[["user có confidence nửa đầu", "trùng nơi (% trong số có cả hai)"]].round(1)
        print(f"{kind} · {name(k)}: {n} user có kết quả ở cả hai nửa, tương quan hạng {rho:.2f}")
    print(f"Bảng T3 ({kind})")
    display(pd.concat(out, axis=1))

home · v1 · validation: 40 user có kết quả ở cả hai nửa, tương quan hạng 0.40
home · v1 · test: 25 user có kết quả ở cả hai nửa, tương quan hạng 0.14
home · v2 · validation: 40 user có kết quả ở cả hai nửa, tương quan hạng 0.43
home · v2 · test: 25 user có kết quả ở cả hai nửa, tương quan hạng 0.25
Bảng T3 (home)


v1 · validation  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                   15   
70–90%                                      4   
từ 90%                                     21   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       20.0   
70–90%                                         25.0   
từ 90%                                         66.7   

                                    v1 · test  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                    8   
70–90%                                      4   
từ 90%                                     13   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       37.5   
70–90%                                        100.0   
từ 90%                                         69.2   

                              v2 · validation  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                    9   
70–90%                                      8   
từ 90%                                     23   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       33.3   
70–90%                                         62.5   
từ 90%                                         78.3   

                                    v2 · test                                   
                   user có confidence nửa đầu trùng nơi (% trong số có cả hai)  
confidence nửa đầu                                                              
dưới 70%                                    6                             33.3  
70–90%                                      2                            100.0  
từ 90%                                     17                             82.4

office · v1 · validation: 20 user có kết quả ở cả hai nửa, tương quan hạng 0.40
office · v1 · test: 8 user có kết quả ở cả hai nửa, tương quan hạng 0.52
office · v2 · validation: 20 user có kết quả ở cả hai nửa, tương quan hạng 0.28
office · v2 · test: 5 user có kết quả ở cả hai nửa, tương quan hạng 0.00
Bảng T3 (office)


v1 · validation  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                    6   
70–90%                                      5   
từ 90%                                     14   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       25.0   
70–90%                                         80.0   
từ 90%                                         81.8   

                                    v1 · test  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                    4   
70–90%                                      0   
từ 90%                                      7   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       50.0   
70–90%                                          NaN   
từ 90%                                         83.3   

                              v2 · validation  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                    3   
70–90%                                      2   
từ 90%                                     19   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       66.7   
70–90%                                         50.0   
từ 90%                                         86.7   

                                    v2 · test                                   
                   user có confidence nửa đầu trùng nơi (% trong số có cả hai)  
confidence nửa đầu                                                              
dưới 70%                                    3                            100.0  
70–90%                                      0                              NaN  
từ 90%                                      6                             75.0

**Đọc kết quả:** trên validation, mức confidence cao có trùng nhiều hơn mức thấp không (ghi số validation sau khi chạy notebook 13). Test cần cho cùng chiều, ở cả v1 lẫn v2. Mẫu test nhỏ nên chỉ đọc chiều, không đọc chênh vài điểm.

## T4. Stay nhận nhầm theo nhãn phương tiện

Giống bảng 4 của notebook 10: % thời gian di chuyển có nhãn bị nhận là stay, trên các user test có `labels.txt`. Bước này nằm trước khi gom địa điểm nên **không phụ thuộc v1 hay v2**.

In [5]:
XE = {"bus", "car", "taxi", "train", "subway", "airplane", "boat", "motorcycle"}
lab_all = set(interim.list_stems(interim.INTERIM_DIR / "labels")) & set(interim.list_stems(Path(ss.CLEAN_DIR)))
lab = {"validation": sorted(lab_all & set(validation)), "test": sorted(lab_all & set(test))}

def bang_nham(users):
    F = sa.run_labels(users, sa.CMP, verbose=False)
    F = F[F["variant"] == "h1_mean"].copy()
    F["nhóm"] = np.where(F["mode"] == "walk", "walk", np.where(F["mode"].isin(XE), "xe", "khác"))
    g = F.groupby("nhóm")[["hours", "false_hours"]].sum()
    out = {"số user có nhãn": len(users), "tổng (%)": 100 * F["false_hours"].sum() / F["hours"].sum()}
    out.update({f"  {k} (%)": 100 * g.at[k, "false_hours"] / g.at[k, "hours"] for k in ("walk", "xe", "khác") if k in g.index})
    return pd.Series(out)

print("Bảng T4. % thời gian di chuyển bị nhận là stay (neo h1_mean)")
display(pd.DataFrame({k: bang_nham(v) for k, v in lab.items()}).round(2))

Bảng T4. % thời gian di chuyển bị nhận là stay (neo h1_mean)


,validation,test
số user có nhãn,42.00,23.00
tổng (%),11.39,10.92
walk (%),26.31,29.52
xe (%),3.02,2.16
khác (%),7.10,5.37


## T5. v1 và v2 trên cùng nhóm: cùng chỗ hay khác chỗ, gộp nhầm mới

Với mỗi user, so home (và office) của v1 với của v2: "cùng chỗ" nếu hai tâm cách nhau ≤ `SAME_M` (cùng hằng số notebook 14). "Gộp mới ở v2" là v2 có cờ `home_eq_office` mà v1 không có (nối chuỗi, như user 073 ở validation). Cột đường kính là của nơi home ở v2. Không có nhãn nên chỉ biết hai bản đồng ý hay không, không biết bản nào đúng (A13).

In [6]:
SAME_M = pe.SAME_M

def centre(places, pid):
    if pd.isna(pid):
        return None
    r = places.loc[places["place_id"] == int(pid)]
    return (float(r["lat_mean"].iloc[0]), float(r["lon_mean"].iloc[0])) if len(r) else None

def so_v1_v2(group, users):
    a, b = ho[("v1", group)].set_index("user"), ho[("v2", group)].set_index("user")
    rows = []
    for u in users:
        if u not in a.index or u not in b.index:
            continue
        p1, p2 = common.load_places(u, "v1")[0], common.load_places(u, "v2")[0]
        row = {"user": u}
        for kind in ("home", "office"):
            c1, c2 = centre(p1, a.at[u, kind]), centre(p2, b.at[u, kind])
            if c1 is None and c2 is None:
                row[kind] = "cả hai không có"
            elif c1 is None or c2 is None:
                row[kind] = "chỉ một bản có"
            else:
                row[kind] = "cùng chỗ" if haversine_m(c1[0], c1[1], c2[0], c2[1]) <= SAME_M else "khác chỗ"
        f1, f2 = str(a.at[u, "flags"]) if pd.notna(a.at[u, "flags"]) else "", str(b.at[u, "flags"]) if pd.notna(b.at[u, "flags"]) else ""
        row["gộp mới ở v2"] = ("home_eq_office" in f2) and ("home_eq_office" not in f1)
        h2 = b.at[u, "home"]
        row["đường kính home v2 (m)"] = round(float(p2.loc[p2["place_id"] == int(h2), "diam_m"].iloc[0]), 0) if pd.notna(h2) and (p2["place_id"] == int(h2)).any() else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("user")

S5 = {"validation": so_v1_v2("validation", validation), "test": so_v1_v2("test", test)}
for kind in ("home", "office"):
    print(f"Bảng T5a ({kind}). Số user theo mức đồng thuận v1 và v2")
    display(pd.DataFrame({g: s[kind].value_counts() for g, s in S5.items()}).fillna(0).astype(int))
print("Bảng T5b. Gộp home với office mới ở v2 (user) và đường kính home v2")
display(pd.DataFrame({g: {"số user": len(s), "gộp mới ở v2": int(s["gộp mới ở v2"].sum()),
                          "đường kính home v2 p95 (m)": s["đường kính home v2 (m)"].quantile(.95)} for g, s in S5.items()}).round(1))
print("Bảng T5c. Các user test bị gộp mới ở v2")
display(S5["test"][S5["test"]["gộp mới ở v2"]])

Bảng T5a (home). Số user theo mức đồng thuận v1 và v2


,validation,test
home,,
cùng chỗ,93,51
cả hai không có,8,2
khác chỗ,7,3


Bảng T5a (office). Số user theo mức đồng thuận v1 và v2


,validation,test
office,,
cùng chỗ,53,30
cả hai không có,52,20
chỉ một bản có,3,6


Bảng T5b. Gộp home với office mới ở v2 (user) và đường kính home v2


,validation,test
số user,108.0,56.0
gộp mới ở v2,3.0,6.0
đường kính home v2 p95 (m),797.9,1124.5


Bảng T5c. Các user test bị gộp mới ở v2


,home,office,gộp mới ở v2,đường kính home v2 (m)
user,,,,
026,cùng chỗ,chỉ một bản có,True,584.0
030,khác chỗ,chỉ một bản có,True,2293.0
036,cùng chỗ,chỉ một bản có,True,653.0
065,cùng chỗ,chỉ một bản có,True,491.0
071,cùng chỗ,chỉ một bản có,True,590.0
092,cùng chỗ,chỉ một bản có,True,580.0


**Đọc kết quả:** trên validation, 7 user có home khác v1 và 3 user bị gộp mới ở v2 (notebook 14, eps = 200 m). Test cần cho cùng cỡ tỉ lệ. Tỉ lệ khác chỗ hoặc gộp mới trên test cao hơn nhiều thì ghi là hạn chế của v2 (nối chuỗi); thấp hơn thì không kết luận gì thêm, vì mẫu test nhỏ.

**Đọc kết quả:** nhận nhầm tổng 10,9% trên test và 11,4% trên validation; walk 29,5% và 26,3%; xe 2,2% và 3,0%. Hai nhóm gần nhau, nên bước tìm stay không chỉ khớp riêng validation.

## Kết luận

Lần chạy test duy nhất (D19), 57 user test, cấu hình khoá ngày 07 và 09/11/2026. Không có nhãn (A13), nên đây là độ ổn định và mức đồng thuận, không phải độ đúng.

**Ổn định, gần validation:**
- Có home: 94,7% (validation 87,0%), cả v1 lẫn v2. Cờ `few_nights` 49,1% và `few_workdays` 49,1%, gần validation (42,6% và 51,3%).
- Nhận nhầm stay theo nhãn phương tiện 10,9% (validation 11,4%).
- Nửa đầu / nửa sau: user có confidence nửa đầu từ 90% trùng nơi nhiều hơn nhóm dưới 70%: home v1 69,2% so với 37,5%, v2 82,4% so với 33,3%. Cùng chiều với validation, nhưng tương quan hạng của home thấp hơn (v1 0,14 so với 0,40; v2 0,25 so với 0,43). Chỉ có 25 user cho home và 5–8 user cho office, nên chỉ đọc chiều, không đọc con số.

**Khác validation, đáng ghi là hạn chế của v2:**
- Gộp home với office mới ở v2 (v2 có `home_eq_office`, v1 không): **6 trong 56 user test (10,7%)**, trên validation là 3 trong 108 user (2,8%). `home_eq_office` của test tăng từ 26,3% (v1) lên 36,8% (v2); validation tăng từ 26,1% lên 28,7%.
- Cả 6 user đó (026, 030, 036, 065, 071, 092) mất office ở v2 mà v1 có. Vì vậy office có ở 52,6% user test với v2, trong khi v1 là 63,2%.
- Đường kính home v2 p95 là 1.125 m trên test (validation 798 m). Ca xa nhất là user 030, đường kính 2.293 m và home khác chỗ so với v1.
- Home khác chỗ giữa v1 và v2 ở 3 user test (5,3%) và 7 user validation (6,1%): tương đương.

**Cách đọc:** chọn `eps` = 200 m trên validation đã đánh giá thấp rủi ro nối chuỗi. Trên nhóm mới, v2 gộp nhầm nhiều gấp khoảng bốn lần, và không có nhãn để biết v1 hay v2 gần với sự thật hơn. Confidence của v2 cao hơn v1 (home 99,8% so với 97,4%) một phần là hiệu ứng của việc gộp cụm, không phải bằng chứng v2 tốt hơn. Mẫu test nhỏ (6 user) nên không nói được tỉ lệ thật.

**Quyết định:** theo D19, không sửa `eps` hay code sau khi xem số test. Giữ hai bản v1 và v2 song song, ghi cảnh báo nối chuỗi của v2 vào tài liệu và slide. Nếu muốn giảm gộp nhầm, đó là một vòng tuning mới trên validation (ví dụ HDBSCAN hoặc `eps` nhỏ hơn) và phải nói rõ số test này không còn là số test sạch.